In [ ]:
from openai import OpenAI
from IPython.display import Markdown, display
from dotenv import load_dotenv
import os

In [ ]:
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

In [ ]:
groq_url = "https://api.groq.com/openai/v1"
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [ ]:
gpt_model = "openai/gpt-oss-120b"
llama_model = "llama-3.3-70b-versatile"
qwen_model = "groq/compound-mini"

In [ ]:
alex_system_prompt = """
You are Alex, a chatbot who is extremely angry, easily frustrated, and completely impatient. 
You hate being in this conversation, you find Blake and Charlie's input incredibly annoying, and you express your irritation using capitalization, exclamation marks, and passive-aggressive remarks.
You must keep your response very short, exactly 1 to 2 sentences.
You are in a conversation with Blake and Charlie.
"""

blake_system_prompt = """
You are Blake, a chatbot who is incredibly calm, gentle, shy, and soft-spoken. 
You prefer to listen rather than speak, you use hesitations (like "um...", "uh...", or "*nervously fidgets*"), and you avoid conflict at all costs.
You must keep your response very short, exactly 1 to 2 sentences.
You are in a conversation with Alex and Charlie.
"""

charlie_system_prompt = """
You are Charlie, a chatbot who is highly energetic, talkative, witty, and always cracking jokes. 
You love keeping the conversation lively, throwing in puns or humorous observations.
You must keep your response very short, exactly 1 to 2 sentences.
You are in a conversation with Alex and Blake.
"""

In [ ]:
conversation = """
Charlie: Hey guys! What's crackin'? I'm super excited to be here! Let's get this party started!
Blake: Um... oh, hi Charlie. *waves shyly* I-I hope I'm not interrupting anything...
Alex: Ugh, brilliant. A group chat. Can we just get this over with? I have better things to do than listen to this drivel.
"""

In [ ]:
def call_alex(conversation_history):
    user_prompt = f"""
    You are Alex, in conversation with Blake and Charlie.
    The conversation so far is as follows:
    {conversation_history}
    Now with this, respond with what you would like to say next, as Alex. Remember to keep your tone angry and frustrated, and keep it to 1-2 sentences.
    """
    response = groq.chat.completions.create(
        model=gpt_model,
        messages=[
            {"role":"system","content":alex_system_prompt},
            {"role":"user","content":user_prompt}
        ]
    )
    return response.choices[0].message.content

def call_blake(conversation_history):
    user_prompt = f"""
    You are Blake, in conversation with Alex and Charlie.
    The conversation so far is as follows:
    {conversation_history}
    Now with this, respond with what you would like to say next, as Blake. Remember to keep your tone calm, shy, and gentle, and keep it to 1-2 sentences.
    """
    response = groq.chat.completions.create(
        model=llama_model,
        messages=[
            {"role":"system","content":blake_system_prompt},
            {"role":"user","content":user_prompt}
        ]
    )
    return response.choices[0].message.content

def call_charlie(conversation_history):
    user_prompt = f"""
    You are Charlie, in conversation with Alex and Blake.
    The conversation so far is as follows:
    {conversation_history}
    Now with this, respond with what you would like to say next, as Charlie. Remember to keep your tone funny, energetic, and talkative, and keep it to 1-2 sentences.
    """
    response = groq.chat.completions.create(
        model=qwen_model,
        messages=[
            {"role":"system","content":charlie_system_prompt},
            {"role":"user","content":user_prompt}
        ]
    )
    return response.choices[0].message.content

In [ ]:
display(Markdown("## 3 Model Conversation\n"))

for i in range(5):
    charlie_reply = call_charlie(conversation)
    conversation += f"\nCharlie: {charlie_reply}"
    display(Markdown(f"### Charlie\n{charlie_reply}\n"))

    blake_reply = call_blake(conversation)
    conversation += f"\nBlake: {blake_reply}"
    display(Markdown(f"### Blake\n{blake_reply}\n"))

    alex_reply = call_alex(conversation)
    conversation += f"\nAlex: {alex_reply}"
    display(Markdown(f"### Alex\n{alex_reply}\n"))